In [0]:
#dbutils.widgets.text("race_results_max_ingestion","")
race_results_max_ingestion=dbutils.widgets.get("race_results_max_ingestion")

In [0]:
class Gold_champion_list():
    main_path="/Volumes/formula1_race/default/formula1/"
    gold_path = "formula1_race_project/gold"
    silver_path = "formula1_race_project/silver"

    def __init__(self,table,max_ingestion_date):
         self.table=table
         self.max_ingestion_date=max_ingestion_date #latest water mark value for race_results table from Gold_race_results class

    def read_input(self):
        from pyspark.sql.functions import max,col,expr,count,to_timestamp,lit,try_to_timestamp
        race_year_list=list()
        #print the latest water mark value
        print(f"max_ingestion_date:{self.max_ingestion_date}")
        print(f"max_ingestion_date:{type(self.max_ingestion_date)}")
        #fetching incremental race_results data from gold table race_results
        if spark.catalog.tableExists("formula1_race.silver.race_results"):
            Incr_race_results_df= (spark.read.table('formula1_race.silver.race_results')
                                  .filter(col('results_ingestion_date')>self.max_ingestion_date))
            #fetching distinct race_year from incremental race_results data
            Incr_race_results_df_list =(Incr_race_results_df
                                   .select(col('race_year')).distinct()
                                   .orderBy(col('race_year').asc()).collect()
                                   )
            
            #fetching incremental race_results data and printing count of records
            print("champion_list:Incr_race_results_df batch count")
            display(Incr_race_results_df.select(count('*')))
            race_year_list=[r.race_year for r in Incr_race_results_df_list]
        #listing of distinct race_years and printing the list
        print(f"race_year_list:{race_year_list}")
        return race_year_list
    
    def apply_transformations(self,race_year_list):
        from pyspark.sql.functions import round,col,broadcast,dense_rank,avg,expr,sum,when,concat,lit,count
        from pyspark.sql.window import Window
        spec_window=Window.partitionBy("race_year").orderBy(col('total_points').desc())

        #fetching only required data from race_results table which is required for aggregation and printing the count of records
        Incr_race_results_df= (spark.read.table('formula1_race.silver.race_results')
                               .filter(col('race_year').isin(race_year_list))
                               )
        print("champion_list:Incr_race_results_df original count")
        display(Incr_race_results_df.select(count('*')))

        #aggregating data as per the requirements and printing out sample year data for clarification
        champions_df=(Incr_race_results_df.groupBy(col("race_year"),col("driver_name"))
                        .agg(sum(col("result_points")).alias("total_points"))
                        .withColumn("position",dense_rank().over(spec_window))
                        .withColumn("winning_position",when(col("position")==1,"Champion").otherwise(concat(lit("P_"),col('position'))))
                        .select(col("race_year"),col("driver_name"),col("total_points"),col("position"),col("winning_position"))
                    )
        display(champions_df.filter(col("driver_name")=="Lewis Hamilton"))
        return champions_df
    
    def write_output(self,apply_tran_df):
        # writing those data into gold layer table by partitioning according to filter approache using dynamic partitionOverwriteMode as true with overwritte mode
        (apply_tran_df.write.option("partitionOverwriteMode", "dynamic")
         .partitionBy("race_year").mode("overwrite")
         .saveAsTable(f"formula1_race.gold.{self.table}"))
        print("final_count_in table")
        display(spark.sql(f'select count(*) from formula1_race.gold.{self.table}'))
        print("Data write into gold champion_list table is Done")

    def process(self):
        print("Started gold-ingestion-champion_list in runing....")
        race_year_list=self.read_input()  #return list of distinct race_years
        apply_tran_df=self.apply_transformations(race_year_list) #return aggregated data
        self.write_output(apply_tran_df)#write data into gold table 


In [0]:
Gold_champion_list_instance = Gold_champion_list("champion_list",race_results_max_ingestion)
Gold_champion_list_instance .process()
print("Successfully Gold_champion_list is ran")